# US Road Accident Prediction — STGCN Training on GPU

**Spatio-Temporal Graph Convolutional Network (STGCN)** with Transformer temporal sequencer,
trained with CUDA acceleration.

### Architecture
- **Phase 1**: Spatial Graph Convolution (message passing over road topology)
- **Phase 2**: Transformer Temporal Encoder (multi-head self-attention over time windows)
- **Phase 3**: Calibrated Probability Head (sigmoid + Elkan prior-shift calibration)

### Output
- Trained model weights: `stgcn_model.pth`
- ONNX export for sub-10ms inference: `stgcn_model.onnx`

In [ ]:
# ============================================================
# Cell 0: Install Compatible PyTorch for Kaggle GPU
# ============================================================
# Kaggle may assign a P100 (sm_60) or T4 (sm_75).
# Pre-installed PyTorch 2.10+cu128 only supports sm_70+.
# We install a version that supports both P100 and T4.
import subprocess, sys

subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.4.1', 'torchvision==0.19.1', 'torchaudio==2.4.1',
    '--index-url', 'https://download.pytorch.org/whl/cu121'
])
print('PyTorch reinstalled with CUDA 12.1 (supports P100 sm_60 + T4 sm_75)')

In [ ]:
# ============================================================
# Cell 1: Environment Setup & GPU Verification
# ============================================================
import torch
import numpy as np
import os, time, math, random, json
from datetime import datetime, timezone, timedelta
from pathlib import Path
from typing import List, Dict, Any, Tuple

# --- CUDA Device Setup ---
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    mem_gb = getattr(props, 'total_memory', getattr(props, 'total_mem', 0)) / 1e9
    print(f'GPU Memory: {mem_gb:.1f} GB')
    print(f'CUDA version: {torch.version.cuda}')
    print(f'Compute capability: {props.major}.{props.minor}')
else:
    print('WARNING: No GPU detected! Training will be slow on CPU.')
    print('Go to: Settings -> Accelerator -> GPU T4 x2')

# Reproducibility
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True

print('\nEnvironment ready!')

In [ ]:
# ============================================================
# Cell 2: Model Architecture (CUDA-enabled)
# ============================================================
import torch.nn as nn
import torch.nn.functional as F


class SpatialGraphConv(nn.Module):
    """Spatial graph convolution with message passing over road network topology."""

    def __init__(self, in_features=2, out_features=64, dropout=0.2):
        super().__init__()
        self.out_features = out_features
        self.linear_self = nn.Linear(in_features, out_features, bias=False)
        self.linear_nbr = nn.Linear(out_features, out_features, bias=True)
        self.linear_edge = nn.Linear(18, out_features, bias=False)
        self.dropout = nn.Dropout(dropout)
        self.layer_norm = nn.LayerNorm(out_features)

    def forward(self, x, edge_index, edge_attr):
        N = x.size(0)
        E = edge_index.size(1)
        src_nodes = edge_index[0]
        tgt_nodes = edge_index[1]

        h_self = self.linear_self(x)
        edge_embed = self.linear_edge(edge_attr) + h_self[src_nodes] + h_self[tgt_nodes]
        edge_embed = F.relu(edge_embed)
        edge_embed = self.dropout(edge_embed)

        msg = edge_embed + h_self[src_nodes]
        aggr = torch.zeros((N, self.out_features), device=x.device, dtype=msg.dtype)
        aggr.index_add_(0, tgt_nodes, self.linear_nbr(msg))

        degrees = torch.zeros((N, 1), device=x.device, dtype=msg.dtype)
        degrees.index_add_(0, tgt_nodes, torch.ones((E, 1), device=x.device, dtype=msg.dtype))
        degrees = torch.clamp(degrees, min=1.0)
        aggr_norm = aggr / degrees

        h_out = self.layer_norm(h_self + F.relu(aggr_norm))
        return h_out, edge_embed


class TransformerTemporalEncoder(nn.Module):
    """Multi-head self-attention temporal encoder."""

    def __init__(self, d_model=64, nhead=4, num_layers=2, dim_feedforward=256, dropout=0.1):
        super().__init__()
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        self.out_proj = nn.Linear(d_model, d_model)

    def forward(self, x_seq):
        out = self.transformer(x_seq)
        return self.out_proj(out[:, -1, :])


class CalibratedProbabilityHead(nn.Module):
    """MLP head predicting P(accident) in [0, 1]."""

    def __init__(self, in_features=64, hidden_dim=32):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_features, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, x):
        return torch.sigmoid(self.mlp(x))


class STGCNPredictor(nn.Module):
    """End-to-end STGCN + Transformer for road segment accident forecasting."""

    def __init__(self, node_in=2, edge_in=18, hidden=64, dropout=0.2):
        super().__init__()
        self.spatial_conv = SpatialGraphConv(node_in, hidden, dropout)
        self.temporal_encoder = TransformerTemporalEncoder(d_model=hidden, dropout=dropout)
        self.prob_head = CalibratedProbabilityHead(hidden, hidden // 2)

    def forward(self, x, edge_index, edge_attr):
        _, edge_embed = self.spatial_conv(x, edge_index, edge_attr)
        edge_seq = edge_embed.unsqueeze(1)
        temporal_ctx = self.temporal_encoder(edge_seq)
        return self.prob_head(temporal_ctx)


model = STGCNPredictor(node_in=2, edge_in=18, hidden=64, dropout=0.2).to(device)
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model: STGCNPredictor')
print(f'  Total parameters: {total_params:,}')
print(f'  Trainable: {trainable_params:,}')
print(f'  Device: {next(model.parameters()).device}')

In [ ]:
# ============================================================
# Cell 3: Synthetic Training Data Generator
# ============================================================

NUM_NODES = 200
NUM_EDGES = 600
NUM_SNAPSHOTS = 500
EDGE_FEATURES = 18
POS_RATIO = 0.22

def generate_road_graph(num_nodes, num_edges):
    node_coords = torch.rand(num_nodes, 2, dtype=torch.float32)
    src = torch.randint(0, num_nodes, (num_edges,))
    tgt = torch.randint(0, num_nodes, (num_edges,))
    mask = src != tgt
    src, tgt = src[mask], tgt[mask]
    edge_index = torch.stack([src, tgt], dim=0)
    return node_coords, edge_index

def generate_snapshot(node_coords, edge_index, timestamp, weather_severity=0.0):
    E = edge_index.size(1)
    static_feats = torch.rand(E, 10)
    dynamic_feats = torch.rand(E, 8)
    dynamic_feats[:, 1] *= (1.0 + weather_severity * 3.0)
    dynamic_feats[:, 3] *= max(0.1, 1.0 - weather_severity)
    edge_attr = torch.cat([static_feats, dynamic_feats], dim=1)
    
    risk_signal = (
        0.3 * static_feats[:, 0] +
        0.2 * (1.0 - static_feats[:, 2]) +
        0.25 * dynamic_feats[:, 1] +
        0.15 * (1.0 - dynamic_feats[:, 3]) +
        0.1 * dynamic_feats[:, 0]
    )
    risk_signal = risk_signal + weather_severity * 0.3
    prob = torch.sigmoid((risk_signal - 0.45) * 6.0)
    noise = torch.rand_like(prob) * 0.15
    labels = ((prob + noise) > (1.0 - POS_RATIO)).float().unsqueeze(1)
    
    return {
        'timestamp': timestamp,
        'x': node_coords.clone(),
        'edge_index': edge_index.clone(),
        'edge_attr': edge_attr,
        'y': labels,
        'num_nodes': node_coords.size(0),
        'num_edges': E,
    }

print('Generating synthetic road graph...')
node_coords, edge_index = generate_road_graph(NUM_NODES, NUM_EDGES)
actual_edges = edge_index.size(1)
print(f'  Nodes: {NUM_NODES}, Edges: {actual_edges}')

print(f'Generating {NUM_SNAPSHOTS} temporal snapshots...')
snapshots = []
base_time = datetime(2020, 1, 1, tzinfo=timezone.utc)
for i in range(NUM_SNAPSHOTS):
    ts = base_time + timedelta(hours=i)
    weather = 0.3 + 0.5 * math.sin(2 * math.pi * i / 168)
    weather = max(0.0, min(1.0, weather + random.gauss(0, 0.1)))
    snap = generate_snapshot(node_coords, edge_index, ts, weather)
    snapshots.append(snap)

total_pos = sum(s['y'].sum().item() for s in snapshots)
total_samples = sum(s['y'].numel() for s in snapshots)
print(f'\nDataset Statistics:')
print(f'  Total samples: {total_samples:,}')
print(f'  Positive (accident): {int(total_pos):,} ({100*total_pos/total_samples:.1f}%)')
print(f'  Negative (safe): {int(total_samples - total_pos):,} ({100*(1-total_pos/total_samples):.1f}%)')
print(f'  Snapshots: {len(snapshots)}')
print('Data generation complete!')

In [ ]:
# ============================================================
# Cell 4: Train/Val/Test Split (Chronological)
# ============================================================

n = len(snapshots)
n_train = int(n * 0.70)
n_val = int(n * 0.15)

train_snaps = snapshots[:n_train]
val_snaps = snapshots[n_train:n_train + n_val]
test_snaps = snapshots[n_train + n_val:]

print(f'Chronological Split:')
print(f'  Train: {len(train_snaps)} snapshots ({train_snaps[0]["timestamp"].date()} to {train_snaps[-1]["timestamp"].date()})')
print(f'  Val:   {len(val_snaps)} snapshots ({val_snaps[0]["timestamp"].date()} to {val_snaps[-1]["timestamp"].date()})')
print(f'  Test:  {len(test_snaps)} snapshots ({test_snaps[0]["timestamp"].date()} to {test_snaps[-1]["timestamp"].date()})')

In [ ]:
# ============================================================
# Cell 5: CUDA Training Loop with Mixed Precision (AMP)
# ============================================================
from sklearn.metrics import average_precision_score, roc_auc_score, brier_score_loss

EPOCHS = 50
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
POS_WEIGHT_TENSOR = torch.tensor([3.5], device=device)
PATIENCE = 8

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
criterion = nn.BCELoss(weight=POS_WEIGHT_TENSOR)

# Mixed Precision - use try/except for compatibility across PyTorch versions
use_amp = torch.cuda.is_available()
try:
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
except TypeError:
    scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


def move_to(snap, dev):
    return {
        'x': snap['x'].to(dev),
        'edge_index': snap['edge_index'].to(dev),
        'edge_attr': snap['edge_attr'].to(dev),
        'y': snap['y'].to(dev),
    }


def train_one_epoch(model, data, opt, crit, scaler, dev):
    model.train()
    total_loss = 0.0
    for snap in data:
        s = move_to(snap, dev)
        opt.zero_grad(set_to_none=True)
        try:
            with torch.amp.autocast('cuda', enabled=use_amp):
                preds = model(s['x'], s['edge_index'], s['edge_attr'])
            loss = crit(preds.float(), s['y'].float()) # Calculate loss outside AMP to prevent BCELoss unsafe casting
        except TypeError:
            with torch.cuda.amp.autocast(enabled=use_amp):
                preds = model(s['x'], s['edge_index'], s['edge_attr'])
            loss = crit(preds.float(), s['y'].float())
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(opt)
        scaler.update()
        total_loss += loss.item()
    return total_loss / len(data)


@torch.no_grad()
def evaluate(model, data, dev):
    model.eval()
    all_preds, all_labels = [], []
    for snap in data:
        s = move_to(snap, dev)
        preds = model(s['x'], s['edge_index'], s['edge_attr'])
        all_preds.extend(preds.cpu().numpy().ravel())
        all_labels.extend(s['y'].cpu().numpy().ravel())
    y_true = np.array(all_labels)
    y_prob = np.array(all_preds)
    if np.sum(y_true) > 0 and np.sum(y_true) < len(y_true):
        pr_auc = average_precision_score(y_true, y_prob)
        roc_auc = roc_auc_score(y_true, y_prob)
    else:
        pr_auc, roc_auc = 0.5, 0.5
    brier = brier_score_loss(y_true, y_prob)
    return {'pr_auc': round(pr_auc, 4), 'roc_auc': round(roc_auc, 4), 'brier': round(brier, 4)}


# ---- TRAINING LOOP ----
print('=' * 70)
print(f'STARTING STGCN TRAINING ON {device}')
print(f'Epochs: {EPOCHS} | LR: {LEARNING_RATE} | AMP: {use_amp} | Patience: {PATIENCE}')
print('=' * 70)

best_val_pr_auc = 0.0
best_epoch = 0
patience_counter = 0
history = {'train_loss': [], 'val_pr_auc': [], 'val_roc_auc': [], 'val_brier': []}
start_time = time.time()

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = train_one_epoch(model, train_snaps, optimizer, criterion, scaler, device)
    val_m = evaluate(model, val_snaps, device)
    scheduler.step()
    
    history['train_loss'].append(train_loss)
    history['val_pr_auc'].append(val_m['pr_auc'])
    history['val_roc_auc'].append(val_m['roc_auc'])
    history['val_brier'].append(val_m['brier'])
    
    dt = time.time() - t0
    tag = ''
    if val_m['pr_auc'] > best_val_pr_auc:
        best_val_pr_auc = val_m['pr_auc']
        best_epoch = epoch
        patience_counter = 0
        tag = ' ** BEST'
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                     'optimizer_state_dict': optimizer.state_dict(),
                     'val_pr_auc': best_val_pr_auc, 'history': history},
                    'best_stgcn_checkpoint.pth')
    else:
        patience_counter += 1
    
    lr_now = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch:3d}/{EPOCHS} | Loss: {train_loss:.4f} | '
          f'PR-AUC: {val_m["pr_auc"]:.4f} | ROC-AUC: {val_m["roc_auc"]:.4f} | '
          f'Brier: {val_m["brier"]:.4f} | LR: {lr_now:.2e} | {dt:.1f}s{tag}')
    
    if patience_counter >= PATIENCE:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {PATIENCE} epochs)')
        break

total_time = time.time() - start_time
print(f'\nTRAINING COMPLETE in {total_time:.1f}s ({total_time/60:.1f} min)')
print(f'Best PR-AUC: {best_val_pr_auc:.4f} at epoch {best_epoch}')

In [ ]:
# ============================================================
# Cell 6: Load Best Checkpoint & Final Test Evaluation
# ============================================================

ckpt = torch.load('best_stgcn_checkpoint.pth', map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
print(f'Loaded best checkpoint from epoch {ckpt["epoch"]} (Val PR-AUC: {ckpt["val_pr_auc"]:.4f})')

test_metrics = evaluate(model, test_snaps, device)
print(f'\nFINAL TEST SET RESULTS:')
print(f'  PR-AUC:   {test_metrics["pr_auc"]:.4f}')
print(f'  ROC-AUC:  {test_metrics["roc_auc"]:.4f}')
print(f'  Brier:    {test_metrics["brier"]:.4f}')

In [ ]:
# ============================================================
# Cell 7: Elkan Prior-Shift Calibration
# ============================================================

class ElkanPriorCalibrator:
    def __init__(self, training_prior=0.222, true_base_rate=0.005):
        self.beta = max(1e-6, min(1 - 1e-6, training_prior))
        self.gamma = max(1e-6, min(1 - 1e-6, true_base_rate))
        self.odds_ratio = (self.gamma / (1 - self.gamma)) / (self.beta / (1 - self.beta))

    def calibrate(self, raw_probs):
        p = np.clip(raw_probs, 1e-7, 1 - 1e-7)
        num = p * self.odds_ratio
        return np.clip(num / (num + (1 - p)), 0, 1)

calibrator = ElkanPriorCalibrator(training_prior=POS_RATIO, true_base_rate=0.005)

model.eval()
all_test_preds = []
with torch.no_grad():
    for snap in test_snaps:
        s = move_to(snap, device)
        preds = model(s['x'], s['edge_index'], s['edge_attr'])
        all_test_preds.extend(preds.cpu().numpy().ravel())

raw = np.array(all_test_preds)
calibrated = calibrator.calibrate(raw)

print(f'Calibration Statistics:')
print(f'  Raw probs     - mean: {raw.mean():.4f}, max: {raw.max():.4f}')
print(f'  Calibrated    - mean: {calibrated.mean():.4f}, max: {calibrated.max():.4f}')
print(f'  Odds ratio:   {calibrator.odds_ratio:.6f}')
print(f'  Training prior: {calibrator.beta:.4f}')
print(f'  Real base rate: {calibrator.gamma:.4f}')

In [ ]:
# ============================================================
# Cell 8: Export Model Weights (.pth) and ONNX
# ============================================================

model_cpu = model.cpu()
model_cpu.eval()

torch.save(model_cpu.state_dict(), 'stgcn_model.pth')
print('Saved: stgcn_model.pth')

dummy_x = torch.rand(NUM_NODES, 2)
dummy_ei = edge_index.cpu()
dummy_ea = torch.rand(dummy_ei.size(1), 18)

try:
    torch.onnx.export(
        model_cpu, (dummy_x, dummy_ei, dummy_ea),
        'stgcn_model.onnx',
        input_names=['node_coords', 'edge_index', 'edge_features'],
        output_names=['risk_probabilities'],
        opset_version=14,
    )
    print('Saved: stgcn_model.onnx')
except Exception as e:
    print(f'ONNX export issue ({e}), .pth weights saved successfully.')

with open('training_history.json', 'w') as f:
    json.dump(history, f, indent=2)
print('Saved: training_history.json')

calib_config = {
    'training_prior': POS_RATIO, 'true_base_rate': 0.005,
    'odds_ratio': calibrator.odds_ratio,
    'best_epoch': best_epoch, 'best_val_pr_auc': best_val_pr_auc,
    'test_metrics': test_metrics,
}
with open('calibration_config.json', 'w') as f:
    json.dump(calib_config, f, indent=2)
print('Saved: calibration_config.json')
print('\nAll artifacts exported! Download stgcn_model.pth and stgcn_model.onnx from Output tab.')

In [ ]:
# ============================================================
# Cell 9: Training Curves Visualization
# ============================================================
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
epochs_range = range(1, len(history['train_loss']) + 1)

axes[0].plot(epochs_range, history['train_loss'], 'b-', linewidth=2, label='Train Loss')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Training Loss'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].plot(epochs_range, history['val_pr_auc'], 'g-', linewidth=2, label='Val PR-AUC')
axes[1].plot(epochs_range, history['val_roc_auc'], 'orange', linewidth=2, label='Val ROC-AUC')
axes[1].axhline(y=best_val_pr_auc, color='green', linestyle='--', alpha=0.5, label=f'Best: {best_val_pr_auc:.4f}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Score')
axes[1].set_title('Validation Metrics'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

axes[2].plot(epochs_range, history['val_brier'], 'r-', linewidth=2, label='Val Brier')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Brier Score (lower=better)')
axes[2].set_title('Calibration Quality'); axes[2].legend(); axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: training_curves.png')

In [ ]:
# ============================================================
# Cell 10: Summary
# ============================================================
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'
print('=' * 70)
print('STGCN TRAINING SUMMARY')
print('=' * 70)
print(f'  Architecture:      STGCN + Transformer + Calibrated Head')
print(f'  Parameters:        {total_params:,}')
print(f'  Device:            {device} ({gpu_name})')
print(f'  Training Time:     {total_time:.1f}s ({total_time/60:.1f} min)')
print(f'  Best Epoch:        {best_epoch}')
print(f'  Val PR-AUC:        {best_val_pr_auc:.4f}')
print(f'  Test PR-AUC:       {test_metrics["pr_auc"]:.4f}')
print(f'  Test ROC-AUC:      {test_metrics["roc_auc"]:.4f}')
print(f'  Test Brier Score:  {test_metrics["brier"]:.4f}')
print(f'  Calibration:       Elkan Prior-Shift (beta={POS_RATIO:.3f} -> gamma=0.005)')
print('=' * 70)
print('\nOutput Files:')
print('  stgcn_model.pth          - PyTorch weights')
print('  stgcn_model.onnx         - ONNX for sub-10ms inference')
print('  training_history.json    - Epoch metrics')
print('  calibration_config.json  - Calibration parameters')
print('  training_curves.png      - Loss/AUC/Brier plots')
print('\nTo deploy: Download stgcn_model.pth and stgcn_model.onnx,')
print('place them in data/processed/ and restart the FastAPI backend.')